In [9]:
import pandas as pd
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import LinearRegression
import pickle
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV
from catboost import CatBoostRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression
import pandas as pd
import numpy as np
# split para modelado
from sklearn.model_selection import train_test_split
# Scaled | Escalado
from sklearn.preprocessing import StandardScaler, MinMaxScaler
# Encoding | Codificación
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn import tree
from sklearn.metrics import accuracy_score
# To save models
import math
import json
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestClassifier
# Feature Selection
from sklearn.feature_selection import f_classif, SelectKBest
from sklearn.datasets import load_iris
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import confusion_matrix
from pickle import dump
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from scipy.stats import randint
import joblib
from contextlib import contextmanager
from scipy.stats import randint, uniform
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error
from catboost import CatBoostRegressor
import optuna
from catboost import CatBoostRegressor
from sklearn.metrics import r2_score

In [3]:
df_prueba = "anthonny"
if df_prueba == "anthonny":
    df = pd.read_csv("../data/processed/df_withinetacolum")
    df = df.sort_values("num_semana").reset_index(drop=True)

    weeks = df["num_semana"].unique()
    cut_w = int(len(weeks) * 0.8)

    train_weeks = weeks[:cut_w]
    test_weeks  = weeks[cut_w:]

    train = df[df["num_semana"].isin(train_weeks)]
    test  = df[df["num_semana"].isin(test_weeks)]

    X_train, y_train = train.drop(columns=["y"]), train["y"]
    X_test,  y_test  = test.drop(columns=["y"]),  test["y"]

else:
    df = pd.read_csv("../data/processed/df_ineta.cvs")
    df = df.sort_values("weekend").reset_index(drop=True) 

    X = df.drop(columns=["weekend"])
    y = df["weekend"]

    cut = int(len(df) * 0.8)
    X_train, X_test = X.iloc[:cut], X.iloc[cut:]
    y_train, y_test = y.iloc[:cut], y.iloc[cut:]
    print ("Trabajaremos con el DF de ineta")

In [5]:
df

,product,num_semana,groups,y,y_lag1,y_lag2,y_lag3,y_lag4,y_lag5,y_lag6,y_lag7,y_lag8
0,artelac fco gotero,1,Oftalmología,0,0,0,0,0,0,0,0,0
1,atenolol,1,Salud Cardiovascular,2,0,0,0,0,0,0,0,0
2,metformina,1,Nutrición y Suplementos,5,0,0,0,0,0,0,0,0
3,colgate periogard 90 gr cre de,1,Higiene y Cuidado Personal,5,0,0,0,0,0,0,0,0
4,treg,1,Salud Cardiovascular,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
18100,quidex gts oft,52,Oftalmología,1,0,1,3,2,2,0,3,3
18101,gen lp carbën activado,52,Nutrición y Suplementos,0,0,1,0,2,0,1,0,2
18102,betacort plus cr dérmica,52,Nutrición y Suplementos,3,5,6,8,3,7,3,3,3
18103,acc lp alcohol puro,52,Material de Curación,1,2,6,3,3,4,7,1,1


In [6]:
cb = CatBoostRegressor(
    loss_function="RMSE",
    random_seed=18,
    iterations=5000,
    learning_rate=0.05,
    depth=8,
    verbose=0,
    allow_writing_files=False
)

cb.fit(
    X_train, y_train,
    cat_features=["product","groups"],
    eval_set=(X_test, y_test),
    early_stopping_rounds=200,
    use_best_model=True
)

In [7]:
model = cb
pred_test = model.predict(X_test)
pred_train = model.predict(X_train)
mse_test  = mean_squared_error(y_test, pred_test)
rmse_test = np.sqrt(mse_test)
r2_test   = r2_score(y_test, pred_test)

print(f"MSE (Error cuadrático medio): {mse_test:.2f}")
print(f"RMSE (Raíz del ECM): {rmse_test:.2f} Cantidad media de error en cuanto a prediccion por producto")
print(f"R² (Coef. determinación): {r2_test:.2f} % de precision")

MSE (Error cuadrático medio): 21.88
RMSE (Raíz del ECM): 4.68 Cantidad media de error en cuanto a prediccion por producto
R² (Coef. determinación): 0.72 % de precision


In [10]:
def objective(trial):
    params = {
        "iterations": 5000, # Bajamos a 2000 para las pruebas; el early stopping hará el resto
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1),
        "depth": trial.suggest_int("depth", 4, 10),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1, 10),
        "random_strength": trial.suggest_float("random_strength", 1, 5),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0, 1),
        "border_count": 128, # Un valor fijo suele ser suficiente para ganar tiempo
        "loss_function": "RMSE",
        "verbose": 0,
        "random_seed": 18
    }
    
    model = CatBoostRegressor(**params)
    
    model.fit(
        X_train, y_train,
        cat_features=["product","groups"],
        eval_set=(X_test, y_test),
        early_stopping_rounds=100, # Si en 100 vueltas no mejora, pasa a la siguiente prueba
        use_best_model=True
    )
    
    preds = model.predict(X_test)
    return r2_score(y_test, preds)

# Crear el estudio y ejecutar
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50) # Con 30 pruebas suele ser suficiente para ver tendencia

print(f"Mejor R2: {study.best_value:.4f}")
print(f"Mejores parámetros: {study.best_params}")


[I 2026-02-04 11:19:42,899] A new study created in memory with name: no-name-52b7f5f1-a453-4b20-a87f-a2304425b035
[I 2026-02-04 11:19:43,796] Trial 0 finished with value: 0.7071167313084885 and parameters: {'learning_rate': 0.049990920726003356, 'depth': 4, 'l2_leaf_reg': 1.2954414745280267, 'random_strength': 4.1936319183345345, 'bagging_temperature': 0.35118942886190907}. Best is trial 0 with value: 0.7071167313084885.
[I 2026-02-04 11:19:45,244] Trial 1 finished with value: 0.7181610260345381 and parameters: {'learning_rate': 0.0145776791341693, 'depth': 4, 'l2_leaf_reg': 6.675881198585401, 'random_strength': 4.29192292040122, 'bagging_temperature': 0.683717424903589}. Best is trial 1 with value: 0.7181610260345381.
[I 2026-02-04 11:19:47,598] Trial 2 finished with value: 0.7153707757480589 and parameters: {'learning_rate': 0.0725924919421191, 'depth': 9, 'l2_leaf_reg': 8.795098555478363, 'random_strength': 4.061755820067621, 'bagging_temperature': 0.4278686736446724}. Best is trial

Mejor R2: 0.7254
Mejores parámetros: {'learning_rate': 0.01383653736112142, 'depth': 10, 'l2_leaf_reg': 5.492534603551152, 'random_strength': 4.96892585593161, 'bagging_temperature': 0.802695197543683}


In [11]:
# 1. Definir el modelo con los parámetros encontrados por Optuna
best_params = study.best_params

model_final = CatBoostRegressor(
    iterations=5000,          # Subimos iteraciones para el entrenamiento final
    loss_function="RMSE",
    random_seed=18,
    verbose=100,              # Para ver el progreso cada 100 pasos
    **best_params,
    allow_writing_files=False         # Esto inserta automáticamente: depth, learning_rate, etc.
)

# 2. Entrenar (usamos early_stopping para no sobreajustar)
model_final.fit(
    X_train, y_train,
    cat_features=["product","groups"],
    eval_set=(X_test, y_test),
    early_stopping_rounds=200,
    use_best_model=True
)

0:	learn: 11.7603887	test: 8.8424931	best: 8.8424931 (0)	total: 17.9ms	remaining: 1m 29s
100:	learn: 7.1142703	test: 5.1489653	best: 5.1489653 (100)	total: 2.41s	remaining: 1m 56s
200:	learn: 6.0335938	test: 4.6523035	best: 4.6523035 (200)	total: 4.41s	remaining: 1m 45s
300:	learn: 5.6839727	test: 4.6586057	best: 4.6389102 (233)	total: 6.39s	remaining: 1m 39s
400:	learn: 5.4659044	test: 4.6848467	best: 4.6389102 (233)	total: 8.54s	remaining: 1m 37s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 4.638910205
bestIteration = 233

Shrink model to first 234 iterations.


In [12]:
pred_test = model_final.predict(X_test)
pred_train = model_final.predict(X_train)
mse_test  = mean_squared_error(y_test, pred_test)
rmse_test = np.sqrt(mse_test)
r2_test   = r2_score(y_test, pred_test)

print(f"MSE (Error cuadrático medio): {mse_test:.2f}")
print(f"RMSE (Raíz del ECM): {rmse_test:.2f} Cantidad media de error en cuanto a prediccion por producto")
print(f"R² (Coef. determinación): {r2_test:.2f} % de precision")

MSE (Error cuadrático medio): 21.52
RMSE (Raíz del ECM): 4.64 Cantidad media de error en cuanto a prediccion por producto
R² (Coef. determinación): 0.72 % de precision


In [13]:
dump(model, open("../models/72_Cat_Boost_Regressor.pkl", "wb"))